In [7]:
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

data_url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv"
df = pd.read_csv(data_url)

days = ['All'] + sorted(df['day'].unique().tolist())
day_widget = widgets.SelectMultiple(
    options=days,
    value=['All'],
    description='Day:'
)

sex_widget = widgets.Dropdown(
    options=['All'] + sorted(df['sex'].unique().tolist()),
    value='All',
    description='Sex:'
)

bill_widget = widgets.FloatRangeSlider(
    value=[df['total_bill'].min(), df['total_bill'].max()],
    min=df['total_bill'].min(),
    max=df['total_bill'].max(),
    step=0.5,
    description='Total Bill:'
)

tip_widget = widgets.FloatRangeSlider(
    value=[df['tip'].min(), df['tip'].max()],
    min=df['tip'].min(),
    max=df['tip'].max(),
    step=0.1,
    description='Tip:'
)

smoker_widget = widgets.Checkbox(
    value=False,
    description='Only smokers'
)

update_button = widgets.Button(description='Refresh')

output = widgets.Output()

def filter_data():
    filtered = df.copy()

    selected_days = list(day_widget.value)
    if 'All' not in selected_days:
        filtered = filtered[filtered['day'].isin(selected_days)]

    if sex_widget.value != 'All':
        filtered = filtered[filtered['sex'] == sex_widget.value]

    filtered = filtered[
        (filtered['total_bill'] >= bill_widget.value[0]) &
        (filtered['total_bill'] <= bill_widget.value[1])
    ]

    filtered = filtered[
        (filtered['tip'] >= tip_widget.value[0]) &
        (filtered['tip'] <= tip_widget.value[1])
    ]

    if smoker_widget.value:
        filtered = filtered[filtered['smoker'] == 'Yes']

    return filtered

def update_analysis(button=None):
    with output:
        clear_output()

        filtered = filter_data()

        if filtered.empty:
            print("Нет данных после фильтрации")
            return

        display(filtered)

        avg_bill = filtered['total_bill'].mean()
        avg_tip = filtered['tip'].mean()
        avg_ratio = (filtered['tip'] / filtered['total_bill']).mean()
        count_orders = len(filtered)

        print(f"Средний total_bill: {avg_bill:.2f}")
        print(f"Средний tip: {avg_tip:.2f}")
        print(f"Среднее отношение tip / total_bill: {avg_ratio:.3f}")
        print(f"Количество заказов: {count_orders}")

        plt.figure()
        plt.hist(filtered['total_bill'], bins=20)
        plt.title('Histogram total_bill')
        plt.xlabel('total_bill')
        plt.ylabel('Frequency')
        plt.show()

        plt.figure()
        plt.hist(filtered['tip'], bins=20)
        plt.title('Histogram tip')
        plt.xlabel('tip')
        plt.ylabel('Frequency')
        plt.show()

        plt.figure()
        filtered.boxplot(column='tip', by='sex')
        plt.title('Boxplot tip by sex')
        plt.suptitle('')
        plt.show()

        plt.figure()
        filtered.boxplot(column='total_bill', by='day')
        plt.title('Boxplot total_bill by day')
        plt.suptitle('')
        plt.show()

        plt.figure()
        plt.scatter(filtered['total_bill'], filtered['tip'])
        plt.title('Scatter total_bill vs tip')
        plt.xlabel('total_bill')
        plt.ylabel('tip')
        plt.show()

update_button.on_click(update_analysis)

ui = widgets.VBox([
    day_widget,
    sex_widget,
    bill_widget,
    tip_widget,
    smoker_widget,
    update_button
])

display(ui, output)

update_analysis()


Output()